# gold.monthly_revenue: Materialized View
> Code is run in the gold_declaratives.py file

In [0]:
%sql
-- Run after building the monthly_revenue pipeline
SELECT * FROM ecommerce.gold.monthly_revenue
ORDER BY revenue_month
LIMIT 10;

# gold.seller_performance: Data Table

In [0]:
from pyspark.sql.functions import (
    col, round, avg, count, countDistinct, try_divide,
    sum as spark_sum, datediff, when
)

orders   = spark.read.table("ecommerce.silver.orders")
items    = spark.read.table("ecommerce.silver.order_items")
sellers  = spark.read.table("ecommerce.silver.sellers")
reviews  = spark.read.table("ecommerce.bronze.order_reviews")

seller_perf = (
    orders
    .join(items, "order_id", "inner")
    .join(reviews.select("order_id", "review_score"), "order_id", "left")
    .groupBy("seller_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(spark_sum(col("price") + col("freight_value")), 2).alias("total_revenue"),
        round(avg("review_score"), 2).alias("avg_review_score"),
        round(avg(
            datediff(
                col("order_delivered_customer_date"),
                col("order_purchase_timestamp")
            )
        ), 1).alias("avg_delivery_days"),
        round(
            try_divide(
                count(when(
                    col("order_delivered_customer_date") <= col("order_estimated_delivery_date"),
                    True
                )) * 100.0,
                count(when(
                    col("order_delivered_customer_date").isNotNull(), True
                ))
            ), 1
        ).alias("on_time_delivery_pct")
    )
    .join(sellers, "seller_id", "left")
)

seller_perf.write.mode("overwrite").option("mergeSchema", True).saveAsTable("ecommerce.gold.seller_performance")
print(f"seller_performance: {spark.read.table('ecommerce.gold.seller_performance').count()} sellers")

In [0]:
# Pick any seller_id from the table
seller_id = spark.read.table("ecommerce.gold.seller_performance").first()["seller_id"]

# Check their total_orders against silver directly
gold_count = spark.read.table("ecommerce.gold.seller_performance") \
    .filter(col("seller_id") == seller_id) \
    .select("total_orders").first()["total_orders"]

silver_count = spark.read.table("ecommerce.silver.order_items") \
    .filter(col("seller_id") == seller_id) \
    .select("order_id").distinct().count()

print(f"Gold: {gold_count} | Silver: {silver_count} | Match: {gold_count == silver_count}")

# gold.live_orders: streaming table
> Code is run in the gold_declaratives.py file

In [0]:
%sql
SELECT * FROM ecommerce.gold.live_orders;